# Headline substitution protocol — parametric runner

Runs the paper's **full headline protocol** (fit rich templates for all 448 heads, solo-cost
selection, MLP tables, position-shuffle control pair with 20-epoch fp32 heals, intact-heal
offset) on either corpus, with the calibration offset and seeds as parameters — so replications
and pre-registered variants are one form away.

`Runtime → Change runtime type → GPU`. A free T4 (fp32) takes ~75 min — about the same as an
M4; an L4/A100 is 2–5× faster. fp32 is deliberate: protocol-matched to the paper.

Repo: https://github.com/keithagroves/how-much-transformer-is-code

In [ ]:
!pip install -q transformers datasets accelerate
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU — set Runtime to GPU')


In [ ]:
#@title Settings { run: "auto" }
CORPUS = "fiction"  #@param ["fiction", "wikitext"]
CALIB_OFFSET = 0  #@param {type:"integer"}
SEED_RND = 7  #@param {type:"integer"}
SEED_HEAL = 11  #@param {type:"integer"}
# Pre-registered replication settings: CALIB_OFFSET=70000, SEED_RND=137, SEED_HEAL=138
print(CORPUS, CALIB_OFFSET, SEED_RND, SEED_HEAL)


In [ ]:
#@title Step 1 · Fetch scripts + corpus
BASE = 'https://raw.githubusercontent.com/keithagroves/how-much-transformer-is-code/main/src'
for f in ['replace_all.py','replace_rich.py','rnd_solo.py','mlp_prosthesis.py','heal_shuffle.py','heal_intact_baseline.py']:
    !wget -q -O {f} {BASE}/{f}
if CORPUS == 'fiction':
    !wget -q -O corpus.txt {BASE}/ministral_corpus.txt
else:
    from datasets import load_dataset
    ds = load_dataset('Salesforce/wikitext', 'wikitext-103-raw-v1', split='train', streaming=True)
    buf, n = [], 0
    for row in ds:
        t = row['text'].strip()
        if len(t) > 80: buf.append(t); n += len(t)
        if n >= 1_400_000: break
    open('corpus.txt','w').write('\n'.join(buf))
import os; print('corpus chars:', os.path.getsize('corpus.txt'))


In [ ]:
#@title Step 2 · Run the chain (fits → solo costs → MLP tables → shuffle pair → offset)
import os
os.environ['SUB_CORPUS'] = 'corpus.txt'
os.environ['SUB_CALIB_OFFSET'] = str(CALIB_OFFSET)
os.environ['SUB_SEED_RND'] = str(SEED_RND)
os.environ['SUB_SEED_HEAL'] = str(SEED_HEAL)
!python -W ignore replace_rich.py 2>&1 | tail -15
print('=== DONE_RICH ===')
!python -W ignore rnd_solo.py 2>&1 | tail -3
print('=== DONE_RND ===')
!python -W ignore mlp_prosthesis.py 2>&1 | tail -8
print('=== DONE_MLP ===')
# MLP guard (pre-registered rule): solo-cheapest 6, but fall back to the middle band
# if the scan picks a layer outside [6, 21] — solo ranking is unstable and early
# layers explode jointly (+3.49 vs +0.36).
import torch
c = torch.load('mlp_solo_costs.pt')['costs']
s = sorted(c, key=lambda l: c[l])[:6]
mlps = ','.join(map(str, sorted(s))) if all(6 <= l <= 21 for l in s) else '9,10,11,12,13,14'
os.environ['SUB_MLPS'] = mlps
print('MLP layers:', mlps)
!python -W ignore heal_shuffle.py
print('=== DONE_SHUFFLE ===')
!python -W ignore heal_intact_baseline.py 2>&1 | tail -5
print('=== DONE_ALL ===')


### Reading the output
- **fair cost** = real-code healed damage minus the intact-heal delta printed at the end.
- **shuffled** must heal far worse than **real**, or the recovery is the heal, not the code.
- Published reference values (fiction, original calibration/seeds): real +0.705, shuffled +2.21,
  offset −0.242. WikiText (middle-6 MLPs): real +0.94, shuffled +1.89, offset −0.269.
- Pre-registered replication criteria (see PREREGISTRATION.md): real within ±0.15 of +0.705;
  shuffled − real ≥ +1.0; offset within ±0.05 of −0.242.
